In [ ]:
import pandas as pd
from detoxify import Detoxify
import os
from tqdm import tqdm
import torch
import os
# Isso "esconde" a GPU 0 do script, forçando ele a usar apenas a GPU 1
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# --- Configuração de Pastas e Arquivos ---
# O "../" faz o código sair da pasta 'notebooks' e ir para a raiz do projeto
INPUT_FILE = '../data/processed/preprocess_text.csv'
OUTPUT_FILE = '../data/processed/dados_toxicidade_reddit_100k.csv'

# NOME DA COLUNA: Verifique se o nome da coluna no seu preprocess_text.csv é esse mesmo
TEXT_COLUMN = 'text_clean'  
LIMIT_ROWS = 100000         
BATCH_SIZE = 256            
# -----------------------------------------

# Verificar se temos GPU disponível
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Usando dispositivo: {device.upper()}")

print(f"🤖 Carregando o modelo Detoxify ('original')...")
model = Detoxify('original', device=device)

print(f"📂 Carregando os primeiros {LIMIT_ROWS} posts de {INPUT_FILE}...")

# Lemos o arquivo. Se der erro de coluna não encontrada, altere o TEXT_COLUMN lá em cima
try:
    df = pd.read_csv(INPUT_FILE, nrows=LIMIT_ROWS, usecols=['id', TEXT_COLUMN])
except ValueError:
    # Caso a coluna de ID ou Texto tenha um nome diferente, ele carrega tudo para você ver
    print("⚠️ Aviso: As colunas exatas não foram encontradas. Carregando tudo...")
    df = pd.read_csv(INPUT_FILE, nrows=LIMIT_ROWS)
    print(f"Colunas disponíveis: {df.columns.tolist()}")

# Tratamento de nulos para evitar erro no modelo
df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna('')
texts_list = df[TEXT_COLUMN].tolist()

print(f"📊 Iniciando análise de toxicidade em {len(texts_list)} textos...")

all_results = []

# Loop de processamento em lotes com barra de progresso detalhada
for i in tqdm(range(0, len(texts_list), BATCH_SIZE), desc="Analisando Toxicidade"):
    batch = texts_list[i : i + BATCH_SIZE]
    
    # O modelo retorna um dicionário com várias categorias
    predictions = model.predict(batch)
    
    # Convertemos o dicionário em um DataFrame temporário
    all_results.append(pd.DataFrame(predictions))

print("\n✅ Processamento concluído. Combinando resultados...")

df_scores = pd.concat(all_results).reset_index(drop=True)
df_final = pd.concat([df, df_scores], axis=1)

# Cria a pasta de saída caso ela não exista por algum motivo
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

print(f"💾 Salvando resultados em {OUTPUT_FILE}...")
df_final.to_csv(OUTPUT_FILE, index=False)

print(f"✨ Etapa concluída! {len(df_final)} linhas analisadas e salvas na pasta processed.")

/home/mateus/reddit/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Usando dispositivo: CUDA
🤖 Carregando o modelo Detoxify ('original')...


Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /home/thbraga/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt
100%|██████████| 418M/418M [01:01<00:00, 7.11MB/s]  
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 681.36it/s]


📂 Carregando os primeiros 100000 posts de ../data/processed/preprocess_text.csv...
📊 Iniciando análise de toxicidade em 100000 textos...


Analisando Toxicidade:   0%|          | 0/391 [00:02<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.39 GiB. GPU 0 has a total capacity of 7.92 GiB of which 1.31 GiB is free. Process 1692945 has 1.25 GiB memory in use. Process 2375613 has 1.31 GiB memory in use. Including non-PyTorch memory, this process has 4.05 GiB memory in use. Of the allocated memory 2.91 GiB is allocated by PyTorch, and 1.04 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import pandas as pd
df = pd.read_csv('../data/processed/toxicidade_perspective_100k.csv')
print(f"Número REAL de posts na tabela: {len(df)}")